# 3. AI 에이전트의 도구 생성

### 1. 인프라 및 데이터 준비
- Kafka 브로커가 실행 중이어야 합니다 (`localhost:9092`).
- `fdata/` 폴더에 파티셔닝된 데이터가 준비되어 있어야 합니다.

docker exec -it kafka kafka-topics --create --topic etching-data --bootstrap-server localhost:9092 --partitions 10 --replication-factor 1

In [1]:
import pandas as pd
import numpy as np
import json
import time
import threading
from confluent_kafka import Producer, Consumer, KafkaException
from collections import deque

# 1. 공통 설정
TOPIC = 'etching-data'
BOOTSTRAP_SERVERS = 'localhost:9092'
ALL_FEATURES = ['250.0', '261.8', '266.6', '272.2', '278.3', '284.6', '288.25', '308.25', '309.25', '324.8', '327.5', '336.98', '364.33', '388.0', '394.4', '395.8', '415.0', '532.6', '544.2', '556.7', '580.0', '611.5', '613.9', '616.3', '618.5', '639.7', '643.2', '644.9', '652.8', '660.0', '669.5', '670.6', '674.0', '725.0', '740.8', '748.5', '753.7', '770.6', '773.2', '781.0', '783.5', '787.5', '791.5', '250.0.1', '261.8.1', '266.6.1', '272.2.1', '278.3.1', '284.6.1', '288.25.1', '308.25.1', '309.25.1', '324.8.1', '327.5.1', '336.98.1', '364.33.1', '388.0.1', '394.4.1', '395.8.1', '415.0.1', '532.6.1', '544.2.1', '556.7.1', '580.0.1', '611.5.1', '613.9.1', '616.3.1', '618.5.1', '639.7.1', '643.2.1', '644.9.1', '652.8.1', '660.0.1', '669.5.1', '670.6.1', '674.0.1', '725.0.1', '740.8.1', '748.5.1', '753.7.1', '770.6.1', '773.2.1', '781.0.1', '783.5.1', '787.5.1', '791.5.1', '250.0.2', '261.8.2', '266.6.2', '272.2.2', '278.3.2', '284.6.2', '288.25.2', '308.25.2', '309.25.2', '324.8.2', '327.5.2', '336.98.2', '364.33.2', '388.0.2', '394.4.2', '395.8.2', '415.0.2', '532.6.2', '544.2.2', '556.7.2', '580.0.2', '611.5.2', '613.9.2', '616.3.2', '618.5.2', '639.7.2', '643.2.2', '644.9.2', '652.8.2', '660.0.2', '669.5.2', '670.6.2', '674.0.2', '725.0.2', '740.8.2', '748.5.2', '753.7.2', '770.6.2', '773.2.2', '781.0.2', '783.5.2', '787.5.2', '791.5.2', 'TIME', 'S1V1', 'S1V2', 'S1V3', 'S1V4', 'S1V5', 'S1I1', 'S1I2', 'S1I3', 'S1I4', 'S1I5', 'S1P1', 'S1P2', 'S1P3', 'S1P4', 'S1P5', 'S2V1', 'S2V2', 'S2V3', 'S2V4', 'S2V5', 'S2I1', 'S2I2', 'S2I3', 'S2I4', 'S2I5', 'S2P1', 'S2P2', 'S2P3', 'S2P4', 'S2P5', 'S3V1', 'S3V2', 'S3V3', 'S3V4', 'S3V5', 'S4V1', 'S4V2', 'S4V3', 'S4V4', 'S4V5', 'S34PV1', 'S34PV2', 'S34PV3', 'S34PV4', 'S34PV5', 'S3I1', 'S3I2', 'S3I3', 'S3I4', 'S3I5', 'S4I1', 'S4I2', 'S4I3', 'S4I4', 'S4I5', 'S34PI1', 'S34PI2', 'S34PI3', 'S34PI4', 'S34PI5', 'S34V1', 'S34V2', 'S34V3', 'S34V4', 'S34V5', 'S34I1', 'S34I2', 'S34I3', 'S34I4', 'S34I5', 'Time.1', 'Step Number', 'BCl3 Flow', 'Cl2 Flow', 'RF Btm Pwr', 'RF Btm Rfl Pwr', 'Endpt A', 'He Press', 'Pressure', 'RF Tuner', 'RF Load', 'RF Phase Err', 'RF Pwr', 'RF Impedance', 'TCP Tuner', 'TCP Phase Err', 'TCP Impedance', 'TCP Top Pwr', 'TCP Rfl Pwr', 'TCP Load', 'Vat Valve']

# 2. LSTMAgent 클래스 정의
class LSTMAgent:
    def __init__(self, eq_id, window_size=30):
        self.eq_id = eq_id
        self.window_size = window_size
        self.latest_cache = {feat: 0.0 for feat in ALL_FEATURES}
        self.buffer = deque(maxlen=window_size)
        self.current_run = None

    def update(self, sensor_type, metrics, run_name):
        if run_name != self.current_run:
            self.current_run = run_name
            self.buffer.clear()
            self.latest_cache = {feat: 0.0 for feat in ALL_FEATURES}

        for k, v in metrics.items():
            if k in self.latest_cache: self.latest_cache[k] = v

        if sensor_type == 'oes':
            row = [self.latest_cache[f] for f in ALL_FEATURES]
            self.buffer.append(row)
            if len(self.buffer) == self.window_size:
                return np.array(self.buffer).reshape(1, self.window_size, -1)
        return None

# 3. Consumer 함수 (별도 쓰레드용)
def start_consumer():
    conf = {
        'bootstrap.servers': BOOTSTRAP_SERVERS,
        'group.id': f'agent_group_{time.time()}',
        'auto.offset.reset': 'latest'
    }
    consumer = Consumer(conf)
    consumer.subscribe([TOPIC])
    agents = {f"EQ_{i:02d}": LSTMAgent(f"EQ_{i:02d}") for i in range(1, 11)}
    
    print("🧠 [Consumer] 수신 대기 시작...")
    try:
        while True:
            msg = consumer.poll(1.0)
            if msg is None: continue
            if msg.error(): continue
            
            data = json.loads(msg.value().decode('utf-8'))
            eq_id = data['equipment_id']
            agent = agents[eq_id]
            input_tensor = agent.update(data['type'], data['metrics'], data['run_name'])
            
            if input_tensor is not None:
                print(f"\n✅ [Consumer] {eq_id} 윈도우 완성! Tensor Shape: {input_tensor.shape}")
            else:
                print(".", end="", flush=True)
    except Exception as e:
        print(f"[Consumer Error] {e}")
    finally:
        consumer.close()

# 4. Producer 함수 (메인 실행용)
def start_producer():
    producer = Producer({'bootstrap.servers': BOOTSTRAP_SERVERS})
    df_mach = pd.read_csv('fdata/MACHINE_10EQ_Partitioned.csv')
    df_rfm = pd.read_csv('fdata/RFM_10EQ_Partitioned.csv')
    df_oes = pd.read_csv('fdata/OES_10EQ_Partitioned.csv')
    
    eq_ids = [f"EQ_{i:02d}" for i in range(1, 11)]
    data_maps = {
        'machine': {eid: df_mach[df_mach['equipment_id'] == eid] for eid in eq_ids},
        'rfm': {eid: df_rfm[df_rfm['equipment_id'] == eid] for eid in eq_ids},
        'oes': {eid: df_oes[df_oes['equipment_id'] == eid] for eid in eq_ids}
    }
    pointers = {eid: {'machine': 0, 'rfm': 0, 'oes': 0} for eid in eq_ids}
    
    print("🚀 [Producer] 데이터 스트리밍 시작...")
    step = 1
    while True:
        any_sent = False
        for eid in eq_ids:
            for s_type in ['machine', 'rfm', 'oes']:
                df = data_maps[s_type][eid]
                idx = pointers[eid][s_type]
                if idx < len(df):
                    row = df.iloc[idx]
                    payload = {
                        "equipment_id": eid,
                        "type": s_type,
                        "metrics": row.drop(['equipment_id', 'Run_Name']).to_dict(),
                        "timestamp": time.time(),
                        "run_name": row['Run_Name']
                    }
                    producer.produce(TOPIC, key=eid, value=json.dumps(payload))
                    pointers[eid][s_type] += 1
                    any_sent = True
        
        if not any_sent: break
        producer.flush()
        if step % 50 == 0: 
            print(f"\n[Producer] Step {step} 전송 완료...")
        step += 1
        time.sleep(0.01)
    print("🏁 [Producer] 스트리밍 종료")

# --- 실행부 ---
# Consumer를 백그라운드 쓰레드로 실행
t = threading.Thread(target=start_consumer, daemon=True)
t.start()

time.sleep(2) # Consumer 준비 시간

# Producer 실행 (메인 쓰레드)
start_producer()

🧠 [Consumer] 수신 대기 시작...
🚀 [Producer] 데이터 스트리밍 시작...

[Producer] Step 50 전송 완료...
......................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................